<a href="https://colab.research.google.com/github/ileeee-sh/SeSAC_29CM_Project/blob/main/29cm_Selenium.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get install -y google-chrome-stable -q || pip install chromedriver-autoinstall -q

# Install Chrome
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -q
!pip install selenium -q

# Version check
!google-chrome --version
!chromedriver --version

Reading package lists...
Building dependency tree...
Reading state information...
E: Unable to locate package google-chrome-stable
  Preparing metadata (setup.py) ... done
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 2 not upgraded.
Need to get 11.2 MB/139 MB of archives.
After this operation, 466 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-data all 2.36.0-3build1 [2,824 B

In [2]:
import time
import csv
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException


PRODUCT_URLS = [
    "https://www.29cm.co.kr/products/2130911"
]

OUTPUT_FILE  = "29cm_reviews.csv"
MAX_PAGES    = None
SCROLL_PAUSE = 1.5


def init_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    options.binary_location = "/usr/bin/google-chrome"
    return webdriver.Chrome(options=options)


def get_product_name(driver):
    selectors = [
        "h1[class*='ProductName']",
        "h2[class*='product-name']",
        "[class*='productName']",
        "[class*='product_name']",
        "h1",
    ]
    for sel in selectors:
        try:
            elem = driver.find_element(By.CSS_SELECTOR, sel)
            text = elem.text.strip()
            if text:
                return text
        except NoSuchElementException:
            continue
    return "상품명 미확인"


def click_review_tab(driver):
    try:
        tab = WebDriverWait(driver, 6).until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[contains(text(),'리뷰')] | //a[contains(text(),'리뷰')]")
            )
        )
        driver.execute_script("arguments[0].click();", tab)
        time.sleep(2)
        print("리뷰 탭 클릭 성공")
    except TimeoutException:
        print(" 리뷰 탭 없음 (자동 노출 방식)")


def scroll_to_reviews(driver):
    try:
        first_card = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "li[data-review-id]"))
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", first_card)
        time.sleep(SCROLL_PAUSE)
    except TimeoutException:
        pass


def parse_star_rating(card):
    for sel in ["[class*='StarRating']", "[class*='star']", "[class*='rating']", "[class*='Rating']"]:
        try:
            elem = card.find_element(By.CSS_SELECTOR, sel)
            label = elem.get_attribute("aria-label") or ""
            nums = "".join(c for c in label if c.isdigit() or c == ".")
            if nums:
                return nums[:3]
        except NoSuchElementException:
            continue
    try:
        filled = card.find_elements(By.CSS_SELECTOR, "svg[class*='fill'], [class*='filled']")
        if filled:
            return str(len(filled))
    except Exception:
        pass
    return ""


def parse_reviews(driver, product_url, product_name):
    reviews = []
    review_cards = driver.find_elements(By.CSS_SELECTOR, "li[data-review-id]")

    for card in review_cards:
        review = {
            "product_name": product_name,
            "option":       "",
            "author":       "",
            "date":         "",
            "rating":       "",
            "review_text":  "",
            "product_url":  product_url,
            "crawled_at":   datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

        review["rating"] = parse_star_rating(card)

        for sel in ["[class*='author']", "[class*='Author']", "[class*='nickname']",
                    "[class*='userName']", "[class*='user_name']", "[class*='writer']"]:
            try:
                text = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                if text:
                    review["author"] = text
                    break
            except NoSuchElementException:
                continue

        for sel in ["time", "[class*='date']", "[class*='Date']", "[class*='created']"]:
            try:
                elem = card.find_element(By.CSS_SELECTOR, sel)
                review["date"] = elem.get_attribute("datetime") or elem.text.strip()
                if review["date"]:
                    break
            except NoSuchElementException:
                continue

        for sel in ["[class*='option']", "[class*='Option']", "[class*='variant']",
                    "[class*='item_option']", "[class*='purchaseOption']"]:
            try:
                text = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                if text:
                    review["option"] = text
                    break
            except NoSuchElementException:
                continue

        for sel in [
            "p.break-all",
            "p[class*='text-primary']",
            "p[class*='review']",
            "[class*='ReviewContent']",
            "[class*='reviewContent']",
            "[class*='reviewText']",
            "[class*='review_text']",
            "[class*='content']",
        ]:
            try:
                text = card.find_element(By.CSS_SELECTOR, sel).text.strip()
                if text:
                    review["review_text"] = text
                    break
            except NoSuchElementException:
                continue

        if not review["review_text"]:
            try:
                paras = card.find_elements(By.TAG_NAME, "p")
                if paras:
                    review["review_text"] = max(paras, key=lambda p: len(p.text)).text.strip()
            except Exception:
                pass

        if any([review["rating"], review["review_text"], review["author"]]):
            reviews.append(review)

    return reviews


def go_to_next_page(driver, page):
    for sel in [
        "button[aria-label='다음 페이지']",
        "button[aria-label='다음']",
        "a[aria-label='다음']",
        "[class*='pagination'] [class*='next']:not([disabled])",
        "[class*='Pagination'] [class*='Next']:not([disabled])",
    ]:
        try:
            btn = driver.find_element(By.CSS_SELECTOR, sel)
            if btn.is_enabled() and btn.is_displayed():
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(SCROLL_PAUSE * 1.5)
                return True
        except NoSuchElementException:
            continue

    try:
        next_num = str(page + 1)
        btn = driver.find_element(
            By.XPATH,
            f"//button[normalize-space(text())='{next_num}'] | //a[normalize-space(text())='{next_num}']"
        )
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(SCROLL_PAUSE * 1.5)
        return True
    except NoSuchElementException:
        pass

    return False


def crawl_product(driver, url):
    print(f"\n크롤링 시작: {url}")
    driver.get(url)
    time.sleep(3)

    product_name = get_product_name(driver)
    print(f"   상품명: {product_name}")

    click_review_tab(driver)
    scroll_to_reviews(driver)

    all_reviews = []
    page = 1

    while True:
        print(f"페이지 {page} 파싱 중")
        reviews = parse_reviews(driver, url, product_name)
        all_reviews.extend(reviews)
        print(f"      → {len(reviews)}개 수집 (누적 {len(all_reviews)}개)")

        if MAX_PAGES and page >= MAX_PAGES:
            print("최대 페이지 도달")
            break

        if not go_to_next_page(driver, page):
            print("마지막 페이지")
            break

        scroll_to_reviews(driver)
        page += 1

    return all_reviews


def save_to_csv(all_reviews, filename):
    if not all_reviews:
        print("저장할 데이터가 없습니다.")
        return

    fieldnames = ["product_name", "option", "author", "date", "rating",
                  "review_text", "product_url", "crawled_at"]

    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_reviews)

    print(f"\nCSV 저장 완료: {filename}  (총 {len(all_reviews)}개 리뷰)")


driver = init_driver()
try:
    all_reviews = []
    for url in PRODUCT_URLS:
        try:
            reviews = crawl_product(driver, url)
            all_reviews.extend(reviews)
        except Exception as e:
            print(f" 오류 발생 ({url}): {e}")
        time.sleep(2)

    save_to_csv(all_reviews, OUTPUT_FILE)
finally:
    driver.quit()


크롤링 시작: https://www.29cm.co.kr/products/2130911
   상품명: 상품명 미확인
 리뷰 탭 없음 (자동 노출 방식)
페이지 1 파싱 중
      → 20개 수집 (누적 20개)
페이지 2 파싱 중
      → 20개 수집 (누적 40개)
페이지 3 파싱 중
      → 20개 수집 (누적 60개)
페이지 4 파싱 중
      → 20개 수집 (누적 80개)
페이지 5 파싱 중
      → 20개 수집 (누적 100개)
페이지 6 파싱 중
      → 20개 수집 (누적 120개)
페이지 7 파싱 중
      → 20개 수집 (누적 140개)
페이지 8 파싱 중
      → 20개 수집 (누적 160개)
페이지 9 파싱 중
      → 20개 수집 (누적 180개)
페이지 10 파싱 중
      → 20개 수집 (누적 200개)
페이지 11 파싱 중
      → 20개 수집 (누적 220개)
페이지 12 파싱 중
      → 20개 수집 (누적 240개)
페이지 13 파싱 중
      → 20개 수집 (누적 260개)
페이지 14 파싱 중
      → 20개 수집 (누적 280개)
페이지 15 파싱 중
      → 20개 수집 (누적 300개)
페이지 16 파싱 중
      → 20개 수집 (누적 320개)
페이지 17 파싱 중
      → 20개 수집 (누적 340개)
페이지 18 파싱 중
      → 20개 수집 (누적 360개)
페이지 19 파싱 중
      → 20개 수집 (누적 380개)
페이지 20 파싱 중
      → 20개 수집 (누적 400개)
페이지 21 파싱 중
      → 20개 수집 (누적 420개)
페이지 22 파싱 중
      → 20개 수집 (누적 440개)
페이지 23 파싱 중
      → 20개 수집 (누적 460개)
페이지 24 파싱 중
      → 20개 수집 (누적 480개)
페이지 25 파싱 중
      → 20개 수집 (누적 500개)
페이지